## Imports & Paths

## Load Combined Dataset

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import numpy as np

# === Paths (using relative paths for GitHub) ===
BASE = Path(".")  # Current directory - adjust based on where notebook runs from
DATASET_FILE = BASE / "HE_Van_Hiele_Dataset" / "combined.csv"
OUT_DIR = BASE / "folds"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Load the combined CSV
combined = pd.read_csv(DATASET_FILE)
print(f"✅ Loaded dataset: {combined.shape[0]} rows, {combined.shape[1]} columns")
print(f"Column names: {combined.columns.tolist()}")
display(combined.head())

## Clean Data & Check Label Distribution

In [ ]:
# === Clean data (remove missing levels, fix types) ===
combined["final_decision"] = combined["final_decision"].astype(str).str.strip()
combined = combined[combined["final_decision"].isin(["1", "2", "3", "4", "5"])]
combined["final_decision"] = combined["final_decision"].astype(int)

# Reset idx if missing
if "idx" not in combined.columns or combined["idx"].isnull().all():
    combined["idx"] = range(len(combined))

combined.reset_index(drop=True, inplace=True)
print("✅ Cleaned dataset:", combined.shape)
print("\n📊 Label Distribution:")
combined["final_decision"].value_counts().sort_index().plot.bar(title="Label Distribution")

## Create Stratified Folds

In [ ]:
# === Create stratified folds ===
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X = combined.index.values
y = combined["final_decision"].values

folds = []
for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    train_df = combined.iloc[train_idx].copy()
    test_df  = combined.iloc[test_idx].copy()

    folds.append((train_df, test_df))
    print(f"Fold {fold_idx}: train={len(train_df)}, test={len(test_df)}")

    # Save CSVs
    train_df.to_csv(OUT_DIR / f"fold_{fold_idx}_train.csv", index=False, encoding="utf-8")
    test_df.to_csv(OUT_DIR / f"fold_{fold_idx}_test.csv", index=False, encoding="utf-8")

print(f"\n✅ Folds saved to: {OUT_DIR}")